In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from wordcloud import WordCloud

pio.templates.default = "plotly_dark"

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "unified_clean.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["text"] = df["text"].fillna("").astype(str)
df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
if "domain" not in df.columns:
    df["domain"] = "unknown"
df["domain"] = df["domain"].fillna("unknown").astype(str)

LABEL_NAMES = {0: "Negative", 1: "Neutral", 2: "Positive"}
LABEL_COLORS = {"Negative": "#ff4b4b", "Neutral": "#ffa500", "Positive": "#00c851"}

df["label_name"] = df["label"].map(LABEL_NAMES).fillna(df["label"].astype(str))
df["review_length"] = df["text"].str.split().str.len()

plotly_figures = {}
matplotlib_figures = {}


In [ ]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())
print("\nSample rows:\n")
print(df.head().to_string(index=False))


In [ ]:
label_order = ["Negative", "Neutral", "Positive"]
label_counts = (
    df["label_name"]
    .value_counts()
    .reindex(label_order, fill_value=0)
    .rename_axis("label_name")
    .reset_index(name="count")
)

fig = px.bar(
    label_counts,
    x="label_name",
    y="count",
    color="label_name",
    color_discrete_map=LABEL_COLORS,
    template="plotly_dark",
    title="Label Distribution"
)
fig.update_layout(showlegend=False, xaxis_title="Label", yaxis_title="Count")
plotly_figures["label_distribution"] = fig
fig.show()


In [ ]:
histogram_colors = {"Negative": "#ff4b4b", "Neutral": "#ffa500", "Positive": "#00c851"}
fig = go.Figure()

for label_name in ["Negative", "Neutral", "Positive"]:
    subset = df.loc[df["label_name"] == label_name, "review_length"]
    fig.add_trace(
        go.Histogram(
            x=subset,
            name=label_name,
            marker_color=histogram_colors[label_name],
            opacity=0.55,
            nbinsx=60,
        )
    )

fig.update_layout(
    template="plotly_dark",
    title="Review Length Distribution by Label",
    barmode="overlay",
    xaxis_title="Review Length (words)",
    yaxis_title="Count"
)
plotly_figures["review_length_histogram"] = fig
fig.show()


In [ ]:
domain_counts = df["domain"].fillna("unknown").value_counts().rename_axis("domain").reset_index(name="count")

fig = px.pie(
    domain_counts,
    names="domain",
    values="count",
    template="plotly_dark",
    title="Domain Distribution"
)
plotly_figures["domain_distribution"] = fig
fig.show()


In [ ]:
def top_words_for_label(label_value: int, top_n: int = 20):
    text_series = df.loc[df["label"] == label_value, "text"].dropna().astype(str)
    counter = Counter(word for text in text_series for word in text.split())
    return counter.most_common(top_n)


positive_words = top_words_for_label(2)
negative_words = top_words_for_label(0)

if not positive_words:
    positive_words = [("no_data", 0)]
if not negative_words:
    negative_words = [("no_data", 0)]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Positive Reviews", "Negative Reviews"))
fig.add_trace(
    go.Bar(
        x=[word for word, _ in positive_words],
        y=[count for _, count in positive_words],
        marker_color="#00c851",
        name="Positive"
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=[word for word, _ in negative_words],
        y=[count for _, count in negative_words],
        marker_color="#ff4b4b",
        name="Negative"
    ),
    row=1,
    col=2,
)

fig.update_layout(
    template="plotly_dark",
    title="Top 20 Words: Positive vs Negative Reviews",
    showlegend=False,
    height=600,
)
fig.update_xaxes(tickangle=45)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
plotly_figures["top_words_positive_vs_negative"] = fig
fig.show()


In [ ]:
positive_text = " ".join(df.loc[df["label"] == 2, "text"].dropna().astype(str))
if not positive_text.strip():
    positive_text = "no positive reviews available"

positive_cloud = WordCloud(
    width=1400,
    height=700,
    background_color="black",
    colormap="Greens",
    collocations=False,
).generate(positive_text)

fig, ax = plt.subplots(figsize=(14, 7), facecolor="black")
ax.imshow(positive_cloud, interpolation="bilinear")
ax.set_title("Positive Review Word Cloud", color="white", fontsize=18)
ax.axis("off")
matplotlib_figures["positive_wordcloud"] = fig
plt.show()


In [ ]:
negative_text = " ".join(df.loc[df["label"] == 0, "text"].dropna().astype(str))
if not negative_text.strip():
    negative_text = "no negative reviews available"

negative_cloud = WordCloud(
    width=1400,
    height=700,
    background_color="black",
    colormap="Reds",
    collocations=False,
).generate(negative_text)

fig, ax = plt.subplots(figsize=(14, 7), facecolor="black")
ax.imshow(negative_cloud, interpolation="bilinear")
ax.set_title("Negative Review Word Cloud", color="white", fontsize=18)
ax.axis("off")
matplotlib_figures["negative_wordcloud"] = fig
plt.show()


In [ ]:
average_length_by_domain = (
    df.groupby("domain", dropna=False)["review_length"]
    .mean()
    .sort_values(ascending=True)
    .reset_index(name="average_review_length")
)

fig = px.bar(
    average_length_by_domain,
    x="average_review_length",
    y="domain",
    orientation="h",
    template="plotly_dark",
    title="Average Review Length per Domain",
    color="average_review_length",
    color_continuous_scale="Viridis"
)
fig.update_layout(xaxis_title="Average Length (words)", yaxis_title="Domain")
plotly_figures["average_review_length_by_domain"] = fig
fig.show()


In [ ]:
save_errors = []

for name, fig in plotly_figures.items():
    output_path = FIGURES_DIR / f"{name}.png"
    try:
        fig.write_image(output_path, scale=2)
        print(f"Saved {output_path}")
    except Exception as exc:
        save_errors.append((name, str(exc)))

for name, fig in matplotlib_figures.items():
    output_path = FIGURES_DIR / f"{name}.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"Saved {output_path}")

if save_errors:
    print("\nSome Plotly PNG exports failed. Install kaleido if it is missing and rerun this cell.")
    for name, error in save_errors:
        print(f"- {name}: {error}")
